In [1]:
# Original Kaggle dataset path from ensemble_co2(6).ipynb execution.
from pathlib import Path
DATASET_ROOT = Path("/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset")
print("Dataset:", DATASET_ROOT)


Dataset: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset


In [2]:
# ==========================================================
# REMNET Stage 2: ENSEMBLE-ONLY classification performance
# MobileNetV2 + ResNet50 + EfficientNet-Lite0 | equal-weight Softmax
# Identical frozen split, model training, hyperparameters and 5 seeds.
# Only REMNET is evaluated and reported (not the constituent models).
# No CodeCarbon, energy or CO2 measurement.
# ==========================================================
import os, sys, subprocess, importlib.util, random, copy, hashlib, json, zipfile, platform
from pathlib import Path
from collections import defaultdict

# Set GPU visibility before importing torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for module, package in [("timm", "timm"), ("docx", "python-docx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import timm
from docx import Document

SEEDS = [7, 49, 112, 345, 476]
BATCH_SIZE, MAX_EPOCHS, PATIENCE, MIN_DELTA = 32, 10, 3, 1e-4
OUT = Path("/kaggle/working/REMNET_Stage2_Performance")
(OUT / "Model_Checkpoints").mkdir(parents=True, exist_ok=True)
(OUT / "Test_Probabilities").mkdir(parents=True, exist_ok=True)

# ----------------------------
# Dataset: original Kaggle input path (already attached)
# ----------------------------
search_roots = [DATASET_ROOT]
for archive in sorted(DATASET_ROOT.rglob("*.zip")):
    target = Path("/kaggle/working/remnet_source") / archive.stem
    if not (target / ".extract_complete").exists():
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                if not (target / entry.filename).resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe ZIP entry: {entry.filename}")
            z.extractall(target)
        (target / ".extract_complete").touch()
    search_roots.append(target)

split_dirs = []
for root in search_roots:
    for parent in [root, *sorted((p for p in root.rglob("*") if p.is_dir()), key=str)]:
        children = {p.name.lower(): p for p in parent.iterdir() if p.is_dir()}
        tr, te = children.get("training") or children.get("train"), children.get("testing") or children.get("test")
        if tr and te and {p.name for p in tr.iterdir() if p.is_dir()} == {p.name for p in te.iterdir() if p.is_dir()}:
            split_dirs.append((tr, te))
if not split_dirs:
    raise FileNotFoundError("Training/Testing class folders were not found in the original Kaggle dataset.")
TRAIN_DIR, TEST_DIR = split_dirs[0]
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
assert len(CLASS_NAMES) == 6, f"Expected 6 classes, got {CLASS_NAMES}"
print("Source Training:", TRAIN_DIR, "\nSource Testing:", TEST_DIR)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Stage 1 cleaning: one representative per identical-image group;
# quarantine groups with conflicting labels, and remove the four
# previously visually confirmed near-duplicates.
# The four exclusions are plain filenames, not an encoded manifest.
visual_exclusions = {
    "Training/metal/metal394.jpg", "Training/paper/paper345.jpg",
    "Training/paper/paper390.jpg", "Training/plastic/plastic137.jpg"
}
by_hash = defaultdict(list)
for partition, root in [("Training", TRAIN_DIR), ("Testing", TEST_DIR)]:
    for cls in CLASS_NAMES:
        for path in sorted((root / cls).iterdir()):
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                by_hash[file_sha256(path)].append({"path": path, "class": cls,
                                      "source": f"{partition}/{cls}/{path.name}",
                                      "original_partition": partition})

cleaned = []
conflicts = 0
for digest, group in by_hash.items():
    if len({item["class"] for item in group}) != 1:
        conflicts += len(group)     # Exclude conflicting-label images entirely.
        continue
    item = min(group, key=lambda x: x["source"])  # Prefer Testing over Training.
    cleaned.append({**item, "sha256": digest})

# The Stage 1 80/20 stratified train/validation split was formed BEFORE
# removing 4 visually confirmed near-duplicates. Sort SHA-256 first to
# reproduce exactly the Stage 1 split without embedding any manifest.
original_test = sorted((x for x in cleaned if x["original_partition"] == "Testing"),
                       key=lambda x: x["source"])
train_pool = sorted((x for x in cleaned if x["original_partition"] == "Training"),
                    key=lambda x: x["sha256"])
if (len(cleaned), len(train_pool), len(original_test), conflicts) != (2910, 2447, 463, 6):
    raise ValueError("Dataset differs from audited Stage 1 source; check Kaggle dataset version. "
                     f"Pre-visual counts: total={len(cleaned)}, pool={len(train_pool)}, "
                     f"test={len(original_test)}, conflicting images={conflicts}.")

training, validation = train_test_split(
    train_pool, test_size=0.2, random_state=42,
    stratify=[x["class"] for x in train_pool])
training = sorted((x for x in training if x["source"] not in visual_exclusions),
                  key=lambda x: x["source"])
validation = sorted((x for x in validation if x["source"] not in visual_exclusions),
                    key=lambda x: x["source"])
assert (len(training), len(validation), len(original_test)) == (1956, 487, 463)
assert len({x["sha256"] for x in training + validation + original_test}) == 2906

split_rows = []
for name, group in [("Training", training), ("Validation", validation), ("Testing", original_test)]:
    for item in group:
        split_rows.append({"split": name, "source_path": item["source"],
                           "class": item["class"], "sha256": item["sha256"]})
split_df = pd.DataFrame(split_rows).sort_values(["split", "source_path"])
split_df.to_csv(OUT / "REMNET_Split_Used.csv", index=False)
split_sha = hashlib.sha256(split_df.to_csv(index=False).encode()).hexdigest()
print("Split: Training",len(training),"Validation",len(validation),"Testing",len(original_test))
# Compact integrity check: the original Stage 1 source, members and
# 1,956/487/463 assignments must be IDENTICAL on every execution.
EXPECTED_SPLIT_SHA = "9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792"
if split_sha != EXPECTED_SPLIT_SHA:
    raise RuntimeError("The rebuilt split differs from frozen Stage 1 image membership.")
print("Stage 1 split integrity verified:", split_sha)

# ----------------------------
# Reproducibility, transforms, and datasets
# ----------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU to run the original training configuration.")
print("GPU:", torch.cuda.get_device_name(0), "| Visible GPUs:", torch.cuda.device_count())

train_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
test_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)])

class ImageList(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as im:
            image = self.transform(im.convert("RGB"))
        return image, CLASS_TO_ID[row["class"]]

train_ds = ImageList(training, train_tf)
val_ds = ImageList(validation, test_tf)
test_ds = ImageList(original_test, test_tf)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_model(name, pretrained):
    n = len(CLASS_NAMES)
    if name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "EfficientNet-Lite0":
        m = timm.create_model("efficientnet_lite0", pretrained=pretrained, num_classes=n)
    else:
        raise ValueError(name)
    return m.to(device)

def evaluate_loss(model, loader, criterion):
    model.eval(); total = 0.0
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            total += criterion(logits, labels.to(device)).item() * len(labels)
    return total / len(loader.dataset)

# ----------------------------
# Lightweight pilot: unchanged preprocessing and six-class model output.
# Does not read the held-out test set.
# ----------------------------
if not (OUT / "pilot_ok.json").exists():
    set_seed(7)
    pilot_model = make_model("MobileNetV2", pretrained=True).eval()
    pilot_batch, _ = next(iter(DataLoader(val_ds, batch_size=2, shuffle=False)))
    with torch.no_grad():
        pilot_output = pilot_model(pilot_batch.to(device))
    if tuple(pilot_output.shape) != (2, 6):
        raise RuntimeError("Model output must have 6 class logits")
    (OUT / "pilot_ok.json").write_text(json.dumps({
        "split_sha256": split_sha,
        "gpu": torch.cuda.get_device_name(0)
    }, indent=2))
    del pilot_model
    torch.cuda.empty_cache()
    print("Pilot passed: original data preprocessing and 6-class model output.")
else:
    pilot_info = json.loads((OUT / "pilot_ok.json").read_text())
    if pilot_info["split_sha256"] != split_sha:
        raise RuntimeError("Previous pilot used a different data split")
    print("Reusing pilot record for the identical split.")

# ----------------------------
# Identical constituent training: seed, Adam (1e-4), cross-entropy,
# batch 32, at most 10 epochs, validation-loss checkpoint,
# min_delta=1e-4, patience=3; no modification to learning protocol.
# ----------------------------
MODEL_NAMES = ["MobileNetV2", "ResNet50", "EfficientNet-Lite0"]

def prepare_constituent(name, seed):
    prefix = f"{name}_seed{seed}"
    ckpt_file = OUT / "Model_Checkpoints" / f"{prefix}.pt"
    record_file = OUT / "Model_Checkpoints" / f"{prefix}_training.json"

    if ckpt_file.exists() != record_file.exists():
        raise RuntimeError(f"Incomplete checkpoint/record pair for {prefix}")
    if ckpt_file.exists():
        saved = json.loads(record_file.read_text())
        if (saved.get("split_sha256") != split_sha
                or saved.get("seed") != seed or saved.get("model") != name):
            raise RuntimeError(f"Existing {prefix} belongs to another experiment")
        model = make_model(name, pretrained=False)
        obj = torch.load(ckpt_file, map_location="cpu", weights_only=True)
        if (obj.get("split_sha256") != split_sha
                or obj.get("seed") != seed or obj.get("model") != name):
            raise RuntimeError("Checkpoint metadata mismatch")
        model.load_state_dict(obj["state_dict"], strict=True)
        print(f"{prefix}: validated checkpoint reused without retraining.", flush=True)
        return model.eval(), saved

    model = make_model(name, pretrained=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    best_loss, best_epoch, epochs_run, stale = float("inf"), 0, 0, 0
    best_weights = None
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * len(labels)
            seen += len(labels)
        val_loss = evaluate_loss(model, val_loader, criterion)
        epochs_run = epoch
        print(f"{name}: Epoch {epoch}/{MAX_EPOCHS} | Training Loss: {running_loss/seen:.4f} | Validation Loss: {val_loss:.4f}", flush=True)
        if val_loss < best_loss - MIN_DELTA:
            best_loss, best_epoch, stale = val_loss, epoch, 0
            best_weights = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_weights)
    saved = {"model": name, "seed": seed, "split_sha256": split_sha,
             "epochs_run": epochs_run, "best_epoch": best_epoch,
             "best_val_loss": best_loss}
    torch.save({"state_dict": model.state_dict(), "seed": seed,
                "split_sha256": split_sha, "model": name,
                "best_epoch": best_epoch}, ckpt_file)
    record_file.write_text(json.dumps(saved, indent=2))
    return model.eval(), saved

# ----------------------------
# REMNET ENSEMBLE EVALUATION ONLY
# Reference structure: Ensemble_Acc_SD1 notebook's evaluate_ensemble,
# seed-wise evaluation, then an Ensemble Mean +/- SD display.
# All data preparation and base model training above are unchanged.
# ----------------------------
def evaluate_ensemble(seed, models_for_seed):
    """Score ONLY the equal-weight Softmax ensemble on the held-out test set."""
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    for model in models_for_seed.values():
        model.eval()

    truth_batches = []
    ensemble_probability_batches = []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            # Original REMNET rule: average the 3 per-class Softmax vectors.
            component_probabilities = [
                torch.softmax(models_for_seed[name](images), dim=1)
                for name in MODEL_NAMES
            ]
            ensemble_probability = torch.mean(
                torch.stack(component_probabilities, dim=0), dim=0
            )
            truth_batches.append(labels.cpu().numpy())
            ensemble_probability_batches.append(ensemble_probability.cpu().numpy())

    y_true = np.concatenate(truth_batches)
    p = np.concatenate(ensemble_probability_batches, axis=0)
    expected_truth = np.array([CLASS_TO_ID[row["class"]] for row in original_test])
    if len(y_true) != 463 or not np.array_equal(y_true, expected_truth):
        raise RuntimeError("Test image order/labels differ from the frozen 463-image split")
    if (p.shape != (463, len(CLASS_NAMES)) or not np.isfinite(p).all()
            or not np.allclose(p.sum(axis=1), 1, atol=1e-5)):
        raise RuntimeError(f"Invalid REMNET probabilities for seed {seed}")

    y_pred = p.argmax(axis=1)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=list(range(len(CLASS_NAMES))),
        zero_division=0
    )
    result = {
        "seed": seed,
        "model": "REMNET",
        "test_images": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
    }

    # Optional audit file: ONLY REMNET predictions and 6-class probabilities.
    predicted = {
        "seed": np.full(len(y_true), seed),
        "source_path": [row["source"] for row in original_test],
        "true_class": [CLASS_NAMES[int(idx)] for idx in y_true],
        "true_class_id": y_true,
        "predicted_class": [CLASS_NAMES[int(idx)] for idx in y_pred],
        "predicted_class_id": y_pred,
        "prediction_confidence": p.max(axis=1),
    }
    for class_index, class_name in enumerate(CLASS_NAMES):
        predicted[f"prob_{class_name}"] = p[:, class_index]
    probability_file = (
        OUT / "Test_Probabilities" / f"REMNET_seed{seed}_test_probabilities.csv"
    )
    pd.DataFrame(predicted).to_csv(
        probability_file, index=False, float_format="%.10f"
    )
    return result


# ----------------------------
# MULTI-SEED RUN — base models train as before, but only the
# REMNET ENSEMBLE gets measured and displayed.
# ----------------------------
results = []
for seed in SEEDS:
    print(f"\n=== REMNET Ensemble | Seed {seed} ===", flush=True)
    set_seed(seed)
    models_for_seed = {}
    for name in MODEL_NAMES:
        # Same best-validation-loss model training / checkpoint reload.
        model, _ = prepare_constituent(name, seed)
        models_for_seed[name] = model

    ensemble_result = evaluate_ensemble(seed, models_for_seed)
    results.append(ensemble_result)
    print(
        "REMNET Ensemble -> "
        f"Accuracy: {ensemble_result['accuracy']:.4f}, "
        f"Macro-Precision: {ensemble_result['macro_precision']:.4f}, "
        f"Macro-Recall: {ensemble_result['macro_recall']:.4f}, "
        f"Macro-F1: {ensemble_result['macro_f1']:.4f}",
        flush=True
    )
    del models_for_seed
    torch.cuda.empty_cache()


# ----------------------------
# FINAL REPORT — ONLY REMNET, one ensemble across five seeds.
# Mean +/- SAMPLE SD (ddof=1), as in the previous Stage 2 protocol.
# CSVs store fractions; console/report display percentages.
# ----------------------------
METRIC_COLUMNS = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
results_df = pd.DataFrame(results)[["seed", "model", "test_images"] + METRIC_COLUMNS]
assert len(results_df) == len(SEEDS) == 5
assert results_df["model"].eq("REMNET").all()
assert results_df["seed"].tolist() == SEEDS
results_df.to_csv(
    OUT / "REMNET_Performance_Per_Seed.csv", index=False,
    float_format="%.10f"
)

summary_row = {"model": "REMNET", "seeds": len(SEEDS), "test_images_per_seed": 463}
for metric in METRIC_COLUMNS:
    summary_row[f"{metric}_mean"] = results_df[metric].mean()
    summary_row[f"{metric}_std"] = results_df[metric].std(ddof=1)
summary_df = pd.DataFrame([summary_row])
summary_df.to_csv(
    OUT / "REMNET_Performance_Summary.csv", index=False,
    float_format="%.10f"
)

print("\n" + "=" * 74)
print("REMNET ENSEMBLE — FINAL 5-SEED HELD-OUT TEST PERFORMANCE")
print("Mean +/- sample SD; percentages")
print("=" * 74)
for metric, label in [
    ("accuracy", "Accuracy"),
    ("macro_precision", "Macro-Precision"),
    ("macro_recall", "Macro-Recall"),
    ("macro_f1", "Macro-F1"),
]:
    print(
        f"{label:17s}: {100*summary_row[metric+'_mean']:.2f} "
        f"+/- {100*summary_row[metric+'_std']:.2f}"
    )

# One combined REMNET probability CSV for cross-seed auditing.
combined = pd.concat([
    pd.read_csv(
        OUT / "Test_Probabilities" / f"REMNET_seed{seed}_test_probabilities.csv"
    )
    for seed in SEEDS
], ignore_index=True)
if len(combined) != 463 * len(SEEDS):
    raise RuntimeError("Unexpected REMNET probability row count")
combined.to_csv(
    OUT / "Test_Probabilities" / "REMNET_all_seeds_test_probabilities.csv",
    index=False, float_format="%.10f"
)

configuration = {
    "experiment": "REMNET Stage 2: ensemble-only classification performance",
    "source_dataset": str(DATASET_ROOT),
    "class_names": CLASS_NAMES,
    "train_images": len(train_ds), "validation_images": len(val_ds),
    "test_images": len(test_ds), "test_split_sha256": split_sha,
    "seeds": SEEDS, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE, "min_delta": MIN_DELTA,
    "optimizer": "Adam", "learning_rate": 1e-4,
    "loss_function": "CrossEntropyLoss", "image_size": [32, 32],
    "constituent_models": MODEL_NAMES,
    "reported_model": "REMNET",
    "ensemble": "equal-weight mean of three per-class Softmax probability vectors",
    "metrics": METRIC_COLUMNS,
    "metrics_scale": "fraction in CSV; percent in summary printout/report",
    "macro_average": "unweighted average over six classes; zero_division=0",
    "five_seed_sd": "sample standard deviation, ddof=1",
    "gpu": torch.cuda.get_device_name(0),
    "platform": "Kaggle", "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": __import__("torchvision").__version__,
    "timm": timm.__version__,
}
(OUT / "Experiment_Configuration.json").write_text(
    json.dumps(configuration, indent=2)
)

report = Document()
report.add_heading("REMNET — Stage 2 Ensemble-Only Classification Performance", 0)
report.add_paragraph(
    "Frozen Stage 1: 1,956 training, 487 validation, and 463 held-out test "
    "images. Five seeds: 7, 49, 112, 345, 476. The constituent CNNs "
    "(MobileNetV2, ResNet50 and EfficientNet-Lite0) use the same training "
    "hyperparameters, data transformations and best-validation-loss "
    "checkpoints as the prior Stage 2 performance notebook. Only REMNET "
    "ensemble performance is measured and reported."
)
report.add_paragraph(
    "REMNET is an equal-weight arithmetic mean of the three six-class "
    "Softmax probability vectors, followed by argmax prediction. "
    "The next table contains accuracy, macro-precision, macro-recall, "
    "and macro-F1 as mean +/- sample SD across five seeds, in percent."
)
table = report.add_table(rows=1, cols=5)
for cell, heading in zip(table.rows[0].cells, [
    "Model", "Accuracy (%)", "Macro-Precision (%)", "Macro-Recall (%)", "Macro-F1 (%)"
]):
    cell.text = heading
values = ["REMNET"] + [
    f"{100*summary_row[m+'_mean']:.2f} +/- {100*summary_row[m+'_std']:.2f}"
    for m in METRIC_COLUMNS
]
for cell, value in zip(table.add_row().cells, values):
    cell.text = value
report.add_paragraph(
    "Per-seed REMNET metrics: REMNET_Performance_Per_Seed.csv. "
    "Held-out image probabilities: Test_Probabilities/REMNET_seed*_test_probabilities.csv "
    "and REMNET_all_seeds_test_probabilities.csv. "
    "No base-model performance tables, base-model probability files, "
    "energy measurements, or CO2 reporting are produced by this version."
)
report.save(OUT / "REMNET_Performance_Summary.docx")
print("\nOnly REMNET ensemble performance files saved in:", OUT)
print("Per-seed: REMNET_Performance_Per_Seed.csv")
print("Summary: REMNET_Performance_Summary.csv")
print("Probabilities: Test_Probabilities/REMNET_*.csv")
print("Report: REMNET_Performance_Summary.docx")


Source Training: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Training 
Source Testing: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Testing
Split: Training 1956 Validation 487 Testing 463
Stage 1 split integrity verified: 9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792
GPU: Tesla T4 | Visible GPUs: 1
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 126MB/s]


Pilot passed: original data preprocessing and 6-class model output.

=== REMNET Ensemble | Seed 7 ===
MobileNetV2: Epoch 1/10 | Training Loss: 1.4755 | Validation Loss: 1.1555
MobileNetV2: Epoch 2/10 | Training Loss: 0.8717 | Validation Loss: 0.7409
MobileNetV2: Epoch 3/10 | Training Loss: 0.5509 | Validation Loss: 0.6001
MobileNetV2: Epoch 4/10 | Training Loss: 0.4057 | Validation Loss: 0.4921
MobileNetV2: Epoch 5/10 | Training Loss: 0.3146 | Validation Loss: 0.4237
MobileNetV2: Epoch 6/10 | Training Loss: 0.2409 | Validation Loss: 0.3763
MobileNetV2: Epoch 7/10 | Training Loss: 0.1989 | Validation Loss: 0.3469
MobileNetV2: Epoch 8/10 | Training Loss: 0.1545 | Validation Loss: 0.3449
MobileNetV2: Epoch 9/10 | Training Loss: 0.1466 | Validation Loss: 0.3670
MobileNetV2: Epoch 10/10 | Training Loss: 0.1037 | Validation Loss: 0.3319
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 163MB/s]


ResNet50: Epoch 1/10 | Training Loss: 1.1734 | Validation Loss: 0.6950
ResNet50: Epoch 2/10 | Training Loss: 0.4742 | Validation Loss: 0.4381
ResNet50: Epoch 3/10 | Training Loss: 0.2381 | Validation Loss: 0.3759
ResNet50: Epoch 4/10 | Training Loss: 0.1493 | Validation Loss: 0.3472
ResNet50: Epoch 5/10 | Training Loss: 0.1029 | Validation Loss: 0.4116
ResNet50: Epoch 6/10 | Training Loss: 0.0847 | Validation Loss: 0.3598
ResNet50: Epoch 7/10 | Training Loss: 0.0938 | Validation Loss: 0.3842


model.safetensors: reconstructing file:   0%|          |  0.00B / 18.8MB            

model.safetensors: downloading bytes:           |  0.00B            

EfficientNet-Lite0: Epoch 1/10 | Training Loss: 1.4065 | Validation Loss: 0.9581
EfficientNet-Lite0: Epoch 2/10 | Training Loss: 0.6214 | Validation Loss: 0.7130
EfficientNet-Lite0: Epoch 3/10 | Training Loss: 0.3898 | Validation Loss: 0.6109
EfficientNet-Lite0: Epoch 4/10 | Training Loss: 0.2740 | Validation Loss: 0.5123
EfficientNet-Lite0: Epoch 5/10 | Training Loss: 0.1924 | Validation Loss: 0.4678
EfficientNet-Lite0: Epoch 6/10 | Training Loss: 0.1541 | Validation Loss: 0.4498
EfficientNet-Lite0: Epoch 7/10 | Training Loss: 0.1211 | Validation Loss: 0.4225
EfficientNet-Lite0: Epoch 8/10 | Training Loss: 0.0863 | Validation Loss: 0.4373
EfficientNet-Lite0: Epoch 9/10 | Training Loss: 0.0725 | Validation Loss: 0.4553
EfficientNet-Lite0: Epoch 10/10 | Training Loss: 0.0582 | Validation Loss: 0.4129
REMNET Ensemble -> Accuracy: 0.9460, Macro-Precision: 0.9462, Macro-Recall: 0.9338, Macro-F1: 0.9379

=== REMNET Ensemble | Seed 49 ===
MobileNetV2: Epoch 1/10 | Training Loss: 1.5113 | Val